In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a GPU program that implements top-p (nucleus) sampling for LLM inference.
</p>

<p>
  Top-p sampling is a text generation technique where you sample from the smallest set of tokens whose cumulative probability exceeds threshold p.
  This balances randomness and quality better than pure top-k or greedy sampling.
</p>

<p>
  Given logits (unnormalized scores) from a language model:
  <ol>
    <li>Convert logits to probabilities using softmax</li>
    <li>Sort tokens by probability (descending)</li>
    <li>Find the smallest set where cumulative probability ≥ p (the "nucleus")</li>
    <li>Renormalize the nucleus probabilities to sum to 1</li>
    <li>Sample a token from the nucleus using the provided random seed</li>
  </ol>
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>Ensure numerical stability when computing softmax</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
  logits = [1.0, 2.0, 3.0, 0.5]
  p = 0.9
  seed = 42

Output:
  sampled_token = 2 or 1
  (tokens with highest probabilities, sampled randomly)
</pre>

<h2>Example 2:</h2>
<pre>
Input:
  logits = [10.0, 1.0, 1.0]
  p = 0.5
  seed = 123

Output:
  sampled_token = 0
  (single token dominates the probability mass)
</pre>

<h2>Constraints</h2>
<ul>
  <li>3 &le; <code>vocab_size</code> &le; 50,000</li>
  <li>-100.0 &le; <code>logits[i]</code> &le; 100.0</li>
  <li>0.0 &lt; <code>p</code> &le; 1.0</li>
  <li>0 &le; <code>sampled_token</code> &lt; vocab_size</li>

  <li>Performance is measured with <code>vocab_size</code> = 50,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

extern "C" void solve(const float* logits, const float* p, const int* seed, int* sampled_token,
                      int vocab_size) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


@cute.jit
def solve(
    logits: cute.Tensor,
    p: cute.Tensor,
    seed: cute.Tensor,
    sampled_token: cute.Tensor,
    vocab_size: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


@jax.jit
def solve(logits: jax.Array, p: jax.Array, seed: jax.Array, vocab_size: int) -> jax.Array:
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer
from std.gpu import block_dim, block_idx, thread_idx


@export
def solve(
    logits: UnsafePointer[Float32, MutExternalOrigin],
    p: UnsafePointer[Float32, MutExternalOrigin],
    seed: UnsafePointer[Int32, MutExternalOrigin],
    sampled_token: UnsafePointer[Int32, MutExternalOrigin],
    vocab_size: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


def solve(
    logits: torch.Tensor,
    p: torch.Tensor,
    seed: torch.Tensor,
    sampled_token: torch.Tensor,
    vocab_size: int,
):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


def solve(
    logits: torch.Tensor,
    p: torch.Tensor,
    seed: torch.Tensor,
    sampled_token: torch.Tensor,
    vocab_size: int,
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/60_top_p_sampling/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
